# Monitoring LLM Performance, Costs, and Quality

## Learning Objectives

By completing this notebook, you will:
1. Implement comprehensive monitoring for LLM applications
2. Track cost metrics (tokens, API calls, total spend)
3. Monitor performance metrics (latency, throughput, error rates)
4. Measure output quality with automated metrics
5. Create alerting systems for production issues

## Prerequisites

- Module 4: Notebook 01 completed (API deployment)
- Module 3 completed (Custom applications)
- Understanding of metrics and monitoring concepts
- Familiarity with data visualization

## Estimated Time: 65 minutes

---

## Why Monitor LLMs?

**You can't improve what you don't measure**: Production LLM systems need continuous monitoring.

Key monitoring areas:
- **Cost** - Token usage, API calls, total spend
- **Performance** - Latency, throughput, availability
- **Quality** - Output accuracy, consistency, user satisfaction
- **Usage** - Request patterns, peak times, user behavior
- **Errors** - Failure rates, timeout patterns, degradation

### Key Insight

**LLM costs scale with usage** - Without monitoring, expenses can spiral out of control.

### Monitoring Stack

1. **Metrics collection** - Capture relevant data points
2. **Aggregation** - Summarize over time windows
3. **Visualization** - Dashboards for quick insights
4. **Alerting** - Automated notifications for issues
5. **Analysis** - Deep dives into patterns and trends

## Setup

Import libraries for monitoring and visualization.

In [ ]:
# Purpose: Setup monitoring infrastructure
# Key Concept: Monitoring requires systematic data collection and analysis

import dataiku
from dataiku.llm import LLM
import json
import time
from typing import Dict, List, Optional, Any
from dataclasses import dataclass, asdict, field
from datetime import datetime, timedelta
from collections import defaultdict
import statistics
import logging

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('LLMMonitoring')

print("✓ Monitoring infrastructure ready")

## Metrics Data Models

Define structured metrics for tracking.

In [ ]:
# Purpose: Define metrics data models
# Key Concept: Structured metrics enable systematic analysis

@dataclass
class RequestMetrics:
    """Metrics for a single API request."""
    request_id: str
    timestamp: datetime
    
    # Performance
    latency_ms: float
    tokens_prompt: int
    tokens_completion: int
    tokens_total: int
    
    # Cost
    cost_usd: float
    
    # Quality
    success: bool
    error_type: Optional[str] = None
    
    # Business
    endpoint: str = "unknown"
    user_id: Optional[str] = None

@dataclass
class AggregatedMetrics:
    """Aggregated metrics over a time window."""
    start_time: datetime
    end_time: datetime
    
    # Volume
    total_requests: int
    successful_requests: int
    failed_requests: int
    
    # Performance
    avg_latency_ms: float
    p50_latency_ms: float
    p95_latency_ms: float
    p99_latency_ms: float
    
    # Cost
    total_tokens: int
    total_cost_usd: float
    avg_cost_per_request: float
    
    # Quality
    success_rate: float
    error_rate: float

@dataclass
class QualityMetrics:
    """Quality metrics for LLM outputs."""
    request_id: str
    timestamp: datetime
    
    # Automated quality checks
    format_valid: bool
    schema_compliant: bool
    sentiment_confidence: float
    
    # Human feedback (if available)
    user_rating: Optional[float] = None
    user_feedback: Optional[str] = None

print("✓ Metrics data models defined")

## Metrics Collector

Build a system to collect and store metrics.

In [ ]:
# Purpose: Implement metrics collection system
# Key Concept: Collect metrics at every API call for comprehensive visibility

class MetricsCollector:
    """
    Collect and store metrics for LLM API calls.
    
    In production, this would write to a time-series database
    or logging system like Dataiku's metrics storage.
    """
    
    def __init__(self, pricing_per_1k_tokens: float = 0.008):
        self.pricing = pricing_per_1k_tokens
        self.metrics: List[RequestMetrics] = []
        self.quality_metrics: List[QualityMetrics] = []
    
    def record_request(
        self,
        request_id: str,
        latency_ms: float,
        tokens_prompt: int,
        tokens_completion: int,
        success: bool,
        endpoint: str = "analyze",
        error_type: Optional[str] = None
    ):
        """
        Record metrics for a single request.
        
        This would be called at the end of every API request.
        """
        tokens_total = tokens_prompt + tokens_completion
        cost_usd = (tokens_total / 1000) * self.pricing
        
        metrics = RequestMetrics(
            request_id=request_id,
            timestamp=datetime.now(),
            latency_ms=latency_ms,
            tokens_prompt=tokens_prompt,
            tokens_completion=tokens_completion,
            tokens_total=tokens_total,
            cost_usd=cost_usd,
            success=success,
            error_type=error_type,
            endpoint=endpoint
        )
        
        self.metrics.append(metrics)
        logger.debug(f"Recorded metrics for {request_id}")
    
    def record_quality(
        self,
        request_id: str,
        format_valid: bool,
        schema_compliant: bool,
        sentiment_confidence: float,
        user_rating: Optional[float] = None
    ):
        """Record quality metrics for a request."""
        quality = QualityMetrics(
            request_id=request_id,
            timestamp=datetime.now(),
            format_valid=format_valid,
            schema_compliant=schema_compliant,
            sentiment_confidence=sentiment_confidence,
            user_rating=user_rating
        )
        
        self.quality_metrics.append(quality)
    
    def aggregate(
        self,
        start_time: Optional[datetime] = None,
        end_time: Optional[datetime] = None
    ) -> AggregatedMetrics:
        """
        Aggregate metrics over a time window.
        
        Parameters
        ----------
        start_time : datetime, optional
            Start of time window (default: first metric)
        end_time : datetime, optional
            End of time window (default: last metric)
        
        Returns
        -------
        AggregatedMetrics
            Aggregated statistics
        """
        if not self.metrics:
            raise ValueError("No metrics to aggregate")
        
        # Filter by time window
        filtered = self.metrics
        if start_time:
            filtered = [m for m in filtered if m.timestamp >= start_time]
        if end_time:
            filtered = [m for m in filtered if m.timestamp <= end_time]
        
        if not filtered:
            raise ValueError("No metrics in specified time window")
        
        # Calculate aggregates
        latencies = [m.latency_ms for m in filtered]
        latencies.sort()
        
        total_requests = len(filtered)
        successful = sum(1 for m in filtered if m.success)
        failed = total_requests - successful
        
        return AggregatedMetrics(
            start_time=filtered[0].timestamp,
            end_time=filtered[-1].timestamp,
            total_requests=total_requests,
            successful_requests=successful,
            failed_requests=failed,
            avg_latency_ms=statistics.mean(latencies),
            p50_latency_ms=latencies[int(len(latencies) * 0.5)],
            p95_latency_ms=latencies[int(len(latencies) * 0.95)],
            p99_latency_ms=latencies[int(len(latencies) * 0.99)],
            total_tokens=sum(m.tokens_total for m in filtered),
            total_cost_usd=sum(m.cost_usd for m in filtered),
            avg_cost_per_request=sum(m.cost_usd for m in filtered) / total_requests,
            success_rate=successful / total_requests,
            error_rate=failed / total_requests
        )
    
    def get_recent_metrics(self, minutes: int = 60) -> List[RequestMetrics]:
        """Get metrics from the last N minutes."""
        cutoff = datetime.now() - timedelta(minutes=minutes)
        return [m for m in self.metrics if m.timestamp >= cutoff]

print("✓ Metrics collector implemented")

## Exercise 1: Simulate Production Traffic

**Task**: Simulate a realistic production workload and collect metrics.

Your simulation should include:
- Mix of successful and failed requests
- Varying latencies
- Different token counts
- Quality metrics

In [ ]:
# YOUR CODE HERE

import random

def simulate_production_traffic(collector: MetricsCollector, num_requests: int = 100):
    """
    Simulate production API traffic.
    
    Parameters
    ----------
    collector : MetricsCollector
        Metrics collector to record to
    num_requests : int
        Number of requests to simulate
    """
    print(f"Simulating {num_requests} production requests...\n")
    
    for i in range(num_requests):
        request_id = f"sim_{i:04d}"
        
        # Simulate realistic distributions
        # YOUR CODE HERE
        
        # Latency: mostly fast, occasional slow requests
        if random.random() < 0.05:  # 5% slow requests
            latency_ms = random.uniform(2000, 5000)
        else:
            latency_ms = random.uniform(500, 1500)
        
        # Token counts: vary by request complexity
        tokens_prompt = random.randint(100, 300)
        tokens_completion = random.randint(150, 400)
        
        # Success rate: 95%
        success = random.random() < 0.95
        error_type = None if success else random.choice(['timeout', 'validation', 'llm_error'])
        
        # Record request metrics
        collector.record_request(
            request_id=request_id,
            latency_ms=latency_ms,
            tokens_prompt=tokens_prompt,
            tokens_completion=tokens_completion,
            success=success,
            error_type=error_type
        )
        
        # Record quality metrics for successful requests
        if success:
            collector.record_quality(
                request_id=request_id,
                format_valid=random.random() < 0.98,
                schema_compliant=random.random() < 0.97,
                sentiment_confidence=random.uniform(0.7, 0.99),
                user_rating=random.choice([None, None, None, 4.0, 5.0])  # Sparse ratings
            )
        
        # Simulate time passing
        time.sleep(0.01)
    
    print(f"✓ Simulated {num_requests} requests")

# Run simulation
collector = MetricsCollector(pricing_per_1k_tokens=0.008)
simulate_production_traffic(collector, num_requests=100)

# Display aggregate metrics
agg = collector.aggregate()

print("\n=== Aggregate Metrics ===")
print(f"Total Requests: {agg.total_requests}")
print(f"Success Rate: {agg.success_rate:.1%}")
print(f"Avg Latency: {agg.avg_latency_ms:.0f}ms")
print(f"P95 Latency: {agg.p95_latency_ms:.0f}ms")
print(f"Total Tokens: {agg.total_tokens:,}")
print(f"Total Cost: ${agg.total_cost_usd:.2f}")
print(f"Avg Cost/Request: ${agg.avg_cost_per_request:.4f}")

# Auto-graded checks
assert agg.total_requests == 100, "Should have 100 requests"
assert agg.success_rate > 0.9, "Success rate should be > 90%"
assert agg.total_cost_usd > 0, "Should have non-zero cost"

print("\n✓ Exercise 1 passed!")

## Visualization Dashboard

Create visual dashboards for monitoring.

In [ ]:
# Purpose: Create monitoring visualizations
# Key Concept: Visualizations enable quick identification of issues

class MonitoringDashboard:
    """
    Generate monitoring visualizations.
    """
    
    @staticmethod
    def plot_overview(collector: MetricsCollector):
        """Create overview dashboard with key metrics."""
        fig, axes = plt.subplots(2, 3, figsize=(16, 10))
        fig.suptitle('LLM API Monitoring Dashboard', fontsize=16, fontweight='bold')
        
        metrics = collector.metrics
        
        # 1. Requests over time
        ax1 = axes[0, 0]
        timestamps = [m.timestamp for m in metrics]
        ax1.plot(timestamps, range(1, len(timestamps) + 1), linewidth=2)
        ax1.set_xlabel('Time')
        ax1.set_ylabel('Cumulative Requests')
        ax1.set_title('Request Volume', fontweight='bold')
        ax1.grid(alpha=0.3)
        
        # 2. Latency distribution
        ax2 = axes[0, 1]
        latencies = [m.latency_ms for m in metrics]
        ax2.hist(latencies, bins=30, alpha=0.7, edgecolor='black')
        ax2.axvline(statistics.mean(latencies), color='red', linestyle='--', 
                    label=f'Mean: {statistics.mean(latencies):.0f}ms')
        ax2.set_xlabel('Latency (ms)')
        ax2.set_ylabel('Frequency')
        ax2.set_title('Latency Distribution', fontweight='bold')
        ax2.legend()
        ax2.grid(alpha=0.3)
        
        # 3. Success vs Error rate
        ax3 = axes[0, 2]
        success_count = sum(1 for m in metrics if m.success)
        error_count = len(metrics) - success_count
        ax3.pie([success_count, error_count], 
                labels=['Success', 'Error'],
                autopct='%1.1f%%',
                colors=['#2ecc71', '#e74c3c'],
                startangle=90)
        ax3.set_title('Success Rate', fontweight='bold')
        
        # 4. Token usage over time
        ax4 = axes[1, 0]
        cumulative_tokens = np.cumsum([m.tokens_total for m in metrics])
        ax4.plot(timestamps, cumulative_tokens, linewidth=2, color='orange')
        ax4.set_xlabel('Time')
        ax4.set_ylabel('Cumulative Tokens')
        ax4.set_title('Token Usage', fontweight='bold')
        ax4.grid(alpha=0.3)
        
        # 5. Cost accumulation
        ax5 = axes[1, 1]
        cumulative_cost = np.cumsum([m.cost_usd for m in metrics])
        ax5.plot(timestamps, cumulative_cost, linewidth=2, color='green')
        ax5.set_xlabel('Time')
        ax5.set_ylabel('Cumulative Cost ($)')
        ax5.set_title('Cost Accumulation', fontweight='bold')
        ax5.grid(alpha=0.3)
        
        # 6. Error types
        ax6 = axes[1, 2]
        error_types = defaultdict(int)
        for m in metrics:
            if not m.success and m.error_type:
                error_types[m.error_type] += 1
        
        if error_types:
            ax6.bar(error_types.keys(), error_types.values(), alpha=0.7)
            ax6.set_xlabel('Error Type')
            ax6.set_ylabel('Count')
            ax6.set_title('Error Breakdown', fontweight='bold')
            ax6.tick_params(axis='x', rotation=45)
            ax6.grid(alpha=0.3)
        else:
            ax6.text(0.5, 0.5, 'No Errors', ha='center', va='center', fontsize=14)
            ax6.set_title('Error Breakdown', fontweight='bold')
        
        plt.tight_layout()
        plt.show()
    
    @staticmethod
    def plot_quality_metrics(collector: MetricsCollector):
        """Visualize quality metrics."""
        quality = collector.quality_metrics
        
        if not quality:
            print("No quality metrics available")
            return
        
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        fig.suptitle('Quality Metrics', fontsize=14, fontweight='bold')
        
        # Format compliance
        ax1 = axes[0]
        format_valid = sum(1 for q in quality if q.format_valid)
        ax1.pie([format_valid, len(quality) - format_valid],
                labels=['Valid', 'Invalid'],
                autopct='%1.1f%%',
                colors=['#2ecc71', '#e74c3c'])
        ax1.set_title('Format Compliance')
        
        # Confidence distribution
        ax2 = axes[1]
        confidences = [q.sentiment_confidence for q in quality]
        ax2.hist(confidences, bins=20, alpha=0.7, edgecolor='black')
        ax2.set_xlabel('Confidence Score')
        ax2.set_ylabel('Frequency')
        ax2.set_title('Sentiment Confidence')
        ax2.grid(alpha=0.3)
        
        # User ratings
        ax3 = axes[2]
        ratings = [q.user_rating for q in quality if q.user_rating is not None]
        if ratings:
            rating_counts = defaultdict(int)
            for r in ratings:
                rating_counts[r] += 1
            ax3.bar(rating_counts.keys(), rating_counts.values(), alpha=0.7)
            ax3.set_xlabel('Rating')
            ax3.set_ylabel('Count')
            ax3.set_title(f'User Ratings (n={len(ratings)})')
            ax3.set_ylim(bottom=0)
            ax3.grid(alpha=0.3)
        else:
            ax3.text(0.5, 0.5, 'No Ratings Yet', ha='center', va='center', fontsize=12)
            ax3.set_title('User Ratings')
        
        plt.tight_layout()
        plt.show()

# Generate dashboard
dashboard = MonitoringDashboard()
dashboard.plot_overview(collector)
dashboard.plot_quality_metrics(collector)

print("\n✓ Dashboard generated")

## Alerting System

Implement automated alerts for production issues.

In [ ]:
# Purpose: Implement alerting system
# Key Concept: Proactive alerts prevent small issues from becoming big problems

from enum import Enum

class AlertSeverity(Enum):
    """Alert severity levels."""
    INFO = "info"
    WARNING = "warning"
    ERROR = "error"
    CRITICAL = "critical"

@dataclass
class Alert:
    """Alert notification."""
    severity: AlertSeverity
    title: str
    message: str
    metric_value: float
    threshold: float
    timestamp: datetime = field(default_factory=datetime.now)

class AlertingSystem:
    """
    Monitor metrics and generate alerts based on thresholds.
    
    In production, this would integrate with:
    - Email notifications
    - Slack/Teams webhooks
    - PagerDuty
    - Dataiku scenarios
    """
    
    def __init__(self):
        # Define thresholds
        self.thresholds = {
            'error_rate': {'warning': 0.05, 'critical': 0.10},  # 5%, 10%
            'p95_latency_ms': {'warning': 2000, 'critical': 3000},  # 2s, 3s
            'cost_per_hour': {'warning': 10.0, 'critical': 20.0},  # $10, $20
            'format_compliance': {'warning': 0.95, 'critical': 0.90}  # 95%, 90%
        }
        
        self.alerts: List[Alert] = []
    
    def check_metrics(self, collector: MetricsCollector) -> List[Alert]:
        """
        Check metrics against thresholds and generate alerts.
        
        Returns
        -------
        List[Alert]
            Any alerts triggered
        """
        new_alerts = []
        
        # Get recent metrics (last hour)
        recent = collector.get_recent_metrics(minutes=60)
        if not recent:
            return new_alerts
        
        # Check error rate
        error_rate = sum(1 for m in recent if not m.success) / len(recent)
        alert = self._check_threshold(
            'error_rate',
            error_rate,
            self.thresholds['error_rate'],
            higher_is_worse=True
        )
        if alert:
            new_alerts.append(alert)
        
        # Check P95 latency
        latencies = sorted([m.latency_ms for m in recent])
        p95_latency = latencies[int(len(latencies) * 0.95)]
        alert = self._check_threshold(
            'p95_latency_ms',
            p95_latency,
            self.thresholds['p95_latency_ms'],
            higher_is_worse=True
        )
        if alert:
            new_alerts.append(alert)
        
        # Check cost per hour
        total_cost = sum(m.cost_usd for m in recent)
        # Extrapolate to hourly rate
        time_span_hours = (recent[-1].timestamp - recent[0].timestamp).total_seconds() / 3600
        cost_per_hour = total_cost / time_span_hours if time_span_hours > 0 else 0
        alert = self._check_threshold(
            'cost_per_hour',
            cost_per_hour,
            self.thresholds['cost_per_hour'],
            higher_is_worse=True
        )
        if alert:
            new_alerts.append(alert)
        
        # Check format compliance
        quality = [q for q in collector.quality_metrics 
                  if any(q.request_id == m.request_id for m in recent)]
        if quality:
            format_compliance = sum(1 for q in quality if q.format_valid) / len(quality)
            alert = self._check_threshold(
                'format_compliance',
                format_compliance,
                self.thresholds['format_compliance'],
                higher_is_worse=False  # Lower is worse for compliance
            )
            if alert:
                new_alerts.append(alert)
        
        self.alerts.extend(new_alerts)
        return new_alerts
    
    def _check_threshold(
        self,
        metric_name: str,
        value: float,
        thresholds: Dict[str, float],
        higher_is_worse: bool = True
    ) -> Optional[Alert]:
        """
        Check if metric exceeds thresholds.
        
        Returns
        -------
        Optional[Alert]
            Alert if threshold exceeded, None otherwise
        """
        if higher_is_worse:
            if value >= thresholds['critical']:
                return Alert(
                    severity=AlertSeverity.CRITICAL,
                    title=f"{metric_name} Critical",
                    message=f"{metric_name} is {value:.2f}, exceeds critical threshold {thresholds['critical']}",
                    metric_value=value,
                    threshold=thresholds['critical']
                )
            elif value >= thresholds['warning']:
                return Alert(
                    severity=AlertSeverity.WARNING,
                    title=f"{metric_name} Warning",
                    message=f"{metric_name} is {value:.2f}, exceeds warning threshold {thresholds['warning']}",
                    metric_value=value,
                    threshold=thresholds['warning']
                )
        else:
            if value <= thresholds['critical']:
                return Alert(
                    severity=AlertSeverity.CRITICAL,
                    title=f"{metric_name} Critical",
                    message=f"{metric_name} is {value:.2f}, below critical threshold {thresholds['critical']}",
                    metric_value=value,
                    threshold=thresholds['critical']
                )
            elif value <= thresholds['warning']:
                return Alert(
                    severity=AlertSeverity.WARNING,
                    title=f"{metric_name} Warning",
                    message=f"{metric_name} is {value:.2f}, below warning threshold {thresholds['warning']}",
                    metric_value=value,
                    threshold=thresholds['warning']
                )
        
        return None
    
    def send_alert(self, alert: Alert):
        """
        Send alert notification.
        
        In production, this would:
        - Send email/Slack message
        - Create incident ticket
        - Trigger Dataiku scenario
        """
        severity_icon = {
            AlertSeverity.INFO: 'ℹ️',
            AlertSeverity.WARNING: '⚠️',
            AlertSeverity.ERROR: '❌',
            AlertSeverity.CRITICAL: '🚨'
        }
        
        icon = severity_icon[alert.severity]
        print(f"{icon} [{alert.severity.value.upper()}] {alert.title}")
        print(f"   {alert.message}")
        print(f"   Timestamp: {alert.timestamp.isoformat()}")
        print()

# Test alerting system
alerting = AlertingSystem()
alerts = alerting.check_metrics(collector)

print("=== Alert Check ===")
if alerts:
    print(f"Generated {len(alerts)} alert(s):\n")
    for alert in alerts:
        alerting.send_alert(alert)
else:
    print("✓ No alerts triggered - all metrics healthy")

print("\n✓ Alerting system implemented")

## Exercise 2: Build a Monitoring Report

**Task**: Create a comprehensive monitoring report suitable for stakeholders.

Your report should include:
- Executive summary
- Key metrics table
- Cost analysis
- Quality assessment
- Recommendations

In [ ]:
# YOUR CODE HERE

class MonitoringReport:
    """
    Generate stakeholder-friendly monitoring reports.
    """
    
    def __init__(self, collector: MetricsCollector, alerting: AlertingSystem):
        self.collector = collector
        self.alerting = alerting
    
    def generate_report(self, period_hours: int = 24) -> str:
        """
        Generate comprehensive monitoring report.
        
        Parameters
        ----------
        period_hours : int
            Reporting period in hours
        
        Returns
        -------
        str
            Formatted report
        """
        # YOUR CODE HERE - Build each section
        
        # Get metrics
        recent = self.collector.get_recent_metrics(minutes=period_hours * 60)
        agg = self.collector.aggregate(
            start_time=datetime.now() - timedelta(hours=period_hours)
        )
        
        report = []
        report.append("="*70)
        report.append("LLM API MONITORING REPORT")
        report.append(f"Period: Last {period_hours} hours")
        report.append(f"Generated: {datetime.now().isoformat()}")
        report.append("="*70)
        report.append("")
        
        # Executive Summary
        report.append("EXECUTIVE SUMMARY")
        report.append("-" * 70)
        report.append(f"Total API Calls: {agg.total_requests:,}")
        report.append(f"Success Rate: {agg.success_rate:.1%}")
        report.append(f"Average Latency: {agg.avg_latency_ms:.0f}ms")
        report.append(f"Total Cost: ${agg.total_cost_usd:.2f}")
        report.append("")
        
        # Key Metrics Table
        report.append("KEY METRICS")
        report.append("-" * 70)
        report.append(f"{'Metric':<30} {'Value':<20} {'Status'}")
        report.append("-" * 70)
        
        # Performance
        latency_status = "✓" if agg.p95_latency_ms < 2000 else "⚠"
        report.append(f"{'P95 Latency':<30} {agg.p95_latency_ms:.0f}ms {'':>8} {latency_status}")
        
        # Reliability
        reliability_status = "✓" if agg.success_rate >= 0.95 else "⚠"
        report.append(f"{'Success Rate':<30} {agg.success_rate:.1%} {'':>12} {reliability_status}")
        
        # Cost
        avg_cost_status = "✓" if agg.avg_cost_per_request < 0.01 else "⚠"
        report.append(f"{'Avg Cost/Request':<30} ${agg.avg_cost_per_request:.4f} {'':>9} {avg_cost_status}")
        
        # Tokens
        report.append(f"{'Total Tokens':<30} {agg.total_tokens:,}")
        report.append("")
        
        # Cost Analysis
        report.append("COST ANALYSIS")
        report.append("-" * 70)
        projected_monthly = (agg.total_cost_usd / period_hours) * 24 * 30
        report.append(f"Period Cost: ${agg.total_cost_usd:.2f}")
        report.append(f"Projected Monthly: ${projected_monthly:.2f}")
        report.append(f"Cost per Success: ${agg.total_cost_usd / agg.successful_requests:.4f}")
        report.append("")
        
        # Quality Assessment
        quality = [q for q in self.collector.quality_metrics 
                  if any(q.request_id == m.request_id for m in recent)]
        
        if quality:
            report.append("QUALITY ASSESSMENT")
            report.append("-" * 70)
            format_compliance = sum(1 for q in quality if q.format_valid) / len(quality)
            schema_compliance = sum(1 for q in quality if q.schema_compliant) / len(quality)
            avg_confidence = statistics.mean(q.sentiment_confidence for q in quality)
            
            report.append(f"Format Compliance: {format_compliance:.1%}")
            report.append(f"Schema Compliance: {schema_compliance:.1%}")
            report.append(f"Avg Confidence: {avg_confidence:.2f}")
            
            ratings = [q.user_rating for q in quality if q.user_rating]
            if ratings:
                report.append(f"User Ratings: {statistics.mean(ratings):.1f}/5.0 (n={len(ratings)})")
            report.append("")
        
        # Active Alerts
        recent_alerts = [a for a in self.alerting.alerts 
                        if (datetime.now() - a.timestamp).total_seconds() < period_hours * 3600]
        
        if recent_alerts:
            report.append("ACTIVE ALERTS")
            report.append("-" * 70)
            for alert in recent_alerts:
                report.append(f"[{alert.severity.value.upper()}] {alert.title}")
                report.append(f"  {alert.message}")
            report.append("")
        
        # Recommendations
        report.append("RECOMMENDATIONS")
        report.append("-" * 70)
        
        # YOUR CODE HERE - Add intelligent recommendations based on metrics
        if agg.p95_latency_ms > 2000:
            report.append("• Investigate latency spikes - consider caching or prompt optimization")
        
        if agg.error_rate > 0.05:
            report.append("• Error rate elevated - review error logs and add retry logic")
        
        if projected_monthly > 500:
            report.append("• Monthly costs projected to exceed $500 - review usage patterns")
        
        if quality and format_compliance < 0.95:
            report.append("• Format compliance below target - improve prompt instructions")
        
        report.append("")
        report.append("="*70)
        
        return "\n".join(report)

# Generate report
reporting = MonitoringReport(collector, alerting)
report_text = reporting.generate_report(period_hours=1)

print(report_text)

# Auto-graded checks
assert "EXECUTIVE SUMMARY" in report_text, "Report should include executive summary"
assert "COST ANALYSIS" in report_text, "Report should include cost analysis"
assert "RECOMMENDATIONS" in report_text, "Report should include recommendations"

print("\n✓ Exercise 2 passed!")

## Summary

Congratulations! You've mastered LLM monitoring in production. Key takeaways:

1. **Comprehensive monitoring** - Track cost, performance, and quality
2. **Real-time metrics** - Collect data at every API call
3. **Visualization** - Dashboards enable quick issue identification
4. **Alerting** - Proactive notifications prevent outages
5. **Reporting** - Stakeholder communication with actionable insights

## Production Monitoring Checklist

**Metrics to Track**:
- [ ] Request volume and patterns
- [ ] Success/error rates
- [ ] Latency (average, P95, P99)
- [ ] Token usage and costs
- [ ] Output quality scores
- [ ] Cache hit rates
- [ ] Error types and frequencies

**Alerting**:
- [ ] Error rate > 5% (warning), > 10% (critical)
- [ ] P95 latency > 2s (warning), > 3s (critical)
- [ ] Hourly cost > $10 (warning), > $20 (critical)
- [ ] Quality metrics degrading
- [ ] Service unavailable

**Dashboards**:
- [ ] Real-time overview dashboard
- [ ] Cost tracking dashboard
- [ ] Quality metrics dashboard
- [ ] Error analysis dashboard

**Reporting**:
- [ ] Daily operational summary
- [ ] Weekly stakeholder report
- [ ] Monthly cost analysis
- [ ] Quarterly trend review

## Monitoring Best Practices

1. **Start simple** - Begin with core metrics, expand over time
2. **Set realistic thresholds** - Based on actual usage patterns
3. **Review regularly** - Weekly metric reviews catch trends early
4. **Document baselines** - Know what "normal" looks like
5. **Act on alerts** - Unused alerts become noise
6. **Correlate metrics** - Cost spikes + latency → optimization opportunity
7. **User feedback** - Quantitative + qualitative = complete picture

## Integration with Dataiku

**Metrics Storage**:
- Store metrics in Dataiku datasets
- Use time-series partitioning
- Enable easy querying and analysis

**Scenarios**:
- Trigger on metric thresholds
- Automated report generation
- Alert notifications

**Dashboards**:
- Build in Dataiku dashboard editor
- Share with stakeholders
- Auto-refresh for real-time view

**Webapps**:
- Interactive monitoring interface
- Drill-down analysis
- Custom visualizations

## Course Complete!

You've completed the Dataiku GenAI with LLM Mesh course. You now know how to:

- Connect to LLM providers through LLM Mesh
- Engineer effective prompts
- Build RAG systems with knowledge bases
- Develop custom LLM applications
- Deploy as production APIs
- Monitor performance, costs, and quality

## Next Steps

- Build your own LLM application
- Experiment with different models
- Optimize prompts for your use case
- Deploy to production
- Share your learnings!

**Happy building!** 🚀